#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col, length


In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}

#Reading From Bronze

In [0]:
df = spark.table("workspace.bronze.cust_info")

# Data Transformations

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance (field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name))) 

##Normalization

In [0]:
df = (
    df.withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "M", "Male")
        .when(F.upper(F.col("cst_gndr")) == "F", "Female")
        .otherwise("n/a")
    )
)

##Removing unwanted Rows

In [0]:
df = df.filter(length(col("cst_key")) == 10)

#Renaming The Columns

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

#Write Into Silver Table 

In [0]:
(
    df.write.mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.crm_customers")
)

In [0]:
%sql
select customer_number from workspace.silver.crm_customers where len(customer_number) <> 10